## Open Tools, Open Sky
### Doing LSST Science with Public Rubin Data & LINCC Frameworks
#### Latest update: Jorge I. Zuluaga

## Single object science

In [8]:
from util import *
import requests
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import pyarrow.parquet as pq
from astroquery.jplsbdb import SBDB

# Macros
DEG = np.pi / 180
RAD = 1 / DEG
ecl_coords, gal_coords = ecl_gal()

# Directories
!mkdir -p data figures
fig_dir='figures/'
data_dir='data/'

# Reload util.py routines
%load_ext autoreload 
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Get information about the object

You need the permanent id:

In [9]:
obj = SBDB.query("Jorgezuluaga")
permid = obj["object"]["des"]  
provid = obj["object"].get("prov_des") 
print(f"MPC ids: {permid}  / {provid}")

MPC ids: 347940  / None


Get the observations:

In [12]:
table = pq.read_table(
    "data/obs_sbn_X05_full.parquet",
    columns=['permid','provid','obstime','ra','dec','mag','band','trkid','rmsmag'],
    filters=[("permid", "==", "347940")],  
)
df_obj = table.to_pandas()
print(f"Total number of observations: {len(df_obj)}")

Total number of observations: 21


Convert to data:

In [13]:
df_obj['obstime'] = pd.to_datetime(df_obj['obstime'])
df_obj['mag'] = pd.to_numeric(df_obj['mag'], errors='coerce')
df_obj['rmsmag'] = pd.to_numeric(df_obj['rmsmag'], errors='coerce').fillna(0.05)
df_obj = df_obj.dropna(subset=['mag', 'band', 'obstime']).sort_values('obstime').reset_index(drop=True)
df_obj['night'] = df_obj['obstime'].dt.floor('D')

### Spectral data

Filter information:

In [15]:
wavelengths = {'Lg': 483, 'Lr': 622, 'Li': 755, 'Lz': 869, 'Ly': 971}
solar_color = {'Lg': 0.45, 'Lr': 0.00, 'Li': -0.12, 'Lz': -0.16, 'Ly': -0.18}
band_colors = {'Lg': '#2ca02c', 'Lr': '#d62728', 'Li': '#ff7f0e', 'Lz': '#8c564b', 'Ly': '#e377c2'}
order = ['Lg', 'Lr', 'Li', 'Lz', 'Ly']
short = {'Lg': 'g', 'Lr': 'r', 'Li': 'i', 'Lz': 'z', 'Ly': 'y'}

def wmean(m, s):
    m = np.asarray(m, float)
    s = np.asarray(s, float)
    w = 1.0 / s**2
    return np.sum(w * m) / np.sum(w), 1.0 / np.sqrt(np.sum(w))